In [12]:
!python -m pip install -q --upgrade pip

!python -m pip install -q paddlepaddle==3.2.0 \
  -i https://www.paddlepaddle.org.cn/packages/stable/cpu/

!python -m pip install -q \
  paddleocr \
  fastapi \
  uvicorn \
  python-multipart \
  requests

In [13]:
import paddle
import paddleocr

print("Paddle:", paddle.__version__)
print("PaddleOCR:", paddleocr.__version__)
print("Device:", paddle.device.get_device())

Paddle: 3.2.0
PaddleOCR: 3.7.0
Device: cpu


In [14]:
pip install paddleocr==3.7.0

In [15]:
%%writefile /content/ocr_server.py

import os

# يجب أن يكون قبل استيراد PaddleOCR
os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

import tempfile
import threading
from pathlib import Path
from typing import Optional

import numpy as np
import paddle
import paddleocr as paddleocr_pkg

from fastapi import FastAPI, File, Form, Header, HTTPException, UploadFile
from paddleocr import PaddleOCR


API_TOKEN = os.getenv("OCR_API_TOKEN", "").strip()
MAX_UPLOAD_BYTES = 30 * 1024 * 1024


# ============================================================
# JSON serialization
# ============================================================

def make_jsonable(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, dict):
        return {
            str(k): make_jsonable(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple)):
        return [
            make_jsonable(x)
            for x in obj
        ]

    return obj


def result_object_to_dict(res):
    if hasattr(res, "json"):
        value = res.json

        if callable(value):
            value = value()

        return make_jsonable(value)

    if isinstance(res, dict):
        return make_jsonable(res)

    if hasattr(res, "__dict__"):
        return make_jsonable(res.__dict__)

    return make_jsonable(res)


# ============================================================
# PaddleOCR
# ============================================================

print("Loading PaddleOCR...")


# نستخدم الإعداد الذي نجح فعلياً في النوتبوك الأصلي.
# لا نستخدم fallback إلى موديل آخر لأن هدفنا الحفاظ على نفس pipeline.
ocr = PaddleOCR(
    lang="ar",
    ocr_version="PP-OCRv5",
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=True,
)


# لا نريد تنفيذ predict متزامن مرتين على نفس instance
ocr_lock = threading.Lock()


print("PaddleOCR loaded.")


# ============================================================
# Auth
# ============================================================

def require_auth(authorization: Optional[str]):

    if not API_TOKEN:
        raise HTTPException(
            status_code=500,
            detail="OCR_API_TOKEN is not configured"
        )

    expected = f"Bearer {API_TOKEN}"

    if authorization != expected:
        raise HTTPException(
            status_code=401,
            detail="Unauthorized"
        )


# ============================================================
# FastAPI
# ============================================================

app = FastAPI(
    title="Colab PaddleOCR Inference Server",
    version="1.0.0"
)


@app.get("/health")
def health(
    authorization: Optional[str] = Header(default=None)
):
    require_auth(authorization)

    return {
        "status": "ok",

        "paddle_version":
            getattr(paddle, "__version__", None),

        "paddleocr_version":
            getattr(paddleocr_pkg, "__version__", None),

        "device":
            paddle.device.get_device(),

        "config": {
            "language": "ar",
            "ocr_version": "PP-OCRv5",

            "use_doc_orientation_classify": False,
            "use_doc_unwarping": False,
            "use_textline_orientation": True,
        },

        "expected_models": {
            "textline_orientation":
                "PP-LCNet_x1_0_textline_ori",

            "detector":
                "PP-OCRv5_server_det",

            "recognizer":
                "arabic_PP-OCRv5_mobile_rec",
        },
    }


@app.post("/infer")
async def infer(
    page: int = Form(...),
    file: UploadFile = File(...),
    authorization: Optional[str] = Header(default=None),
):

    require_auth(authorization)

    if page < 1:
        raise HTTPException(
            status_code=422,
            detail="page must be >= 1"
        )

    content = await file.read()

    if not content:
        raise HTTPException(
            status_code=400,
            detail="Empty file"
        )

    if len(content) > MAX_UPLOAD_BYTES:
        raise HTTPException(
            status_code=413,
            detail="Image too large"
        )

    suffix = Path(
        file.filename or "page.png"
    ).suffix.lower()

    if suffix not in {
        ".png",
        ".jpg",
        ".jpeg",
        ".bmp",
        ".tif",
        ".tiff",
        ".webp",
    }:
        suffix = ".png"

    tmp_path = None

    try:

        # نحفظ نفس bytes التي وصلتنا دون أي تعديل.
        with tempfile.NamedTemporaryFile(
            suffix=suffix,
            delete=False
        ) as tmp:

            tmp.write(content)
            tmp_path = tmp.name

        # هذا هو الجزء الوحيد الذي نريد تشغيله في Colab.
        with ocr_lock:

            prediction_result = ocr.predict(
                tmp_path
            )

        # لا نقوم هنا بتطبيع bbox/text.
        # نرسل RAW PaddleOCR result إلى الجهاز المحلي.
        raw_result = [
            result_object_to_dict(res)
            for res in prediction_result
        ]

        return {
            "ok": True,
            "page": page,
            "api": "v3_predict",
            "raw": raw_result,
        }

    except Exception as exc:

        raise HTTPException(
            status_code=500,
            detail=repr(exc)
        )

    finally:

        if tmp_path:
            try:
                os.remove(tmp_path)
            except OSError:
                pass

Overwriting /content/ocr_server.py


In [16]:
import os
import secrets

TOKEN = secrets.token_urlsafe(32)

os.environ["OCR_API_TOKEN"] = TOKEN

print("OCR API TOKEN:")
print(TOKEN)

OCR API TOKEN:
HbxGLDy7iI-YTHQRNL_DH0tjhQ8G6BJiT04Se9sX_Sg


In [17]:
import subprocess
import sys
import time
import requests
import os # Added for better practice, though not strictly needed for this specific change

SERVER_LOG = "/content/ocr_server.log"

# Terminate any existing uvicorn processes that might be holding port 8000
print("Attempting to terminate any existing uvicorn processes...")
try:
    # Use 'pgrep' to find PIDs of uvicorn processes and 'kill' them
    uvicorn_pids = subprocess.check_output(["pgrep", "-f", "uvicorn ocr_server:app"]).decode().strip().split('\n')
    for pid in uvicorn_pids:
        if pid:
            try:
                subprocess.run(["kill", "-9", pid], check=True)
                print(f"Killed uvicorn process with PID: {pid}")
            except subprocess.CalledProcessError:
                print(f"Failed to kill uvicorn process {pid}.")
    time.sleep(1) # Give some time for the port to free up
except subprocess.CalledProcessError:
    print("No existing uvicorn processes found to terminate or pgrep not available.")


log_file = open(
    SERVER_LOG,
    "w"
)


server_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "ocr_server:app",

        "--host",
        "127.0.0.1",

        "--port",
        "8000",

        "--workers",
        "1",
    ],

    cwd="/content",

    stdout=log_file,
    stderr=subprocess.STDOUT,
)


headers = {
    "Authorization":
        f"Bearer {TOKEN}"
}


for _ in range(300):

    try:

        response = requests.get(
            "http://127.0.0.1:8000/health",
            headers=headers,
            timeout=2,
        )

        if response.status_code == 200:

            print(response.json())
            break

    except Exception:
        pass

    time.sleep(1)

else:
    # Ensure log file is closed before reading its content.
    log_file.close()
    print(open(SERVER_LOG).read())

    # Terminate the newly started server process if it didn't start correctly
    server_process.terminate()
    server_process.wait(timeout=5) # Wait for termination

    raise RuntimeError(
        "OCR server did not start correctly"
    )

Attempting to terminate any existing uvicorn processes...
Killed uvicorn process with PID: 1172
{'status': 'ok', 'paddle_version': '3.2.0', 'paddleocr_version': '3.7.0', 'device': 'cpu', 'config': {'language': 'ar', 'ocr_version': 'PP-OCRv5', 'use_doc_orientation_classify': False, 'use_doc_unwarping': False, 'use_textline_orientation': True}, 'expected_models': {'textline_orientation': 'PP-LCNet_x1_0_textline_ori', 'detector': 'PP-OCRv5_server_det', 'recognizer': 'arabic_PP-OCRv5_mobile_rec'}}


In [18]:
!curl -L \
  --output /tmp/cloudflared.deb \
  "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"

!dpkg -i /tmp/cloudflared.deb

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 18.0M  100 18.0M    0     0  19.7M      0 --:--:-- --:--:-- --:--:-- 47.0M
(Reading database ... 122407 files and directories currently installed.)
Preparing to unpack /tmp/cloudflared.deb ...
Unpacking cloudflared (2026.7.3) over (2026.7.3) ...
Setting up cloudflared (2026.7.3) ...
Processing triggers for man-db (2.10.2-1) ...


In [19]:
!pkill -f cloudflared || true

^C


In [20]:
import subprocess
import re
import time

TUNNEL_LOG = "/content/cloudflared.log"

with open(TUNNEL_LOG, "w") as f:
    tunnel_process = subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            "http://127.0.0.1:8000",
        ],
        stdout=f,
        stderr=subprocess.STDOUT,
    )

OCR_BASE_URL = None

for _ in range(60):

    try:
        text = open(
            TUNNEL_LOG,
            encoding="utf-8",
            errors="ignore"
        ).read()
    except Exception:
        text = ""

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        text
    )

    if match:
        OCR_BASE_URL = match.group(0)
        break

    time.sleep(1)

if OCR_BASE_URL is None:
    print(open(TUNNEL_LOG).read())
    raise RuntimeError("Could not obtain Cloudflare URL")

print("NEW OCR SERVER URL:")
print(OCR_BASE_URL)

NEW OCR SERVER URL:
https://carlo-aaa-builders-bowling.trycloudflare.com


In [21]:
import time
import requests

time.sleep(5)

response = requests.get(
    OCR_BASE_URL + "/health",
    headers={
        "Authorization": f"Bearer {TOKEN}"
    },
    timeout=30,
)

print(response.status_code)
print(response.json())

200
{'status': 'ok', 'paddle_version': '3.2.0', 'paddleocr_version': '3.7.0', 'device': 'cpu', 'config': {'language': 'ar', 'ocr_version': 'PP-OCRv5', 'use_doc_orientation_classify': False, 'use_doc_unwarping': False, 'use_textline_orientation': True}, 'expected_models': {'textline_orientation': 'PP-LCNet_x1_0_textline_ori', 'detector': 'PP-OCRv5_server_det', 'recognizer': 'arabic_PP-OCRv5_mobile_rec'}}


In [22]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    headers={
        "Authorization": f"Bearer {TOKEN}"
    },
    timeout=10,
)

print(response.status_code)
print(response.json())

200
{'status': 'ok', 'paddle_version': '3.2.0', 'paddleocr_version': '3.7.0', 'device': 'cpu', 'config': {'language': 'ar', 'ocr_version': 'PP-OCRv5', 'use_doc_orientation_classify': False, 'use_doc_unwarping': False, 'use_textline_orientation': True}, 'expected_models': {'textline_orientation': 'PP-LCNet_x1_0_textline_ori', 'detector': 'PP-OCRv5_server_det', 'recognizer': 'arabic_PP-OCRv5_mobile_rec'}}
